In [1]:
import getml
import mlflow
import getml_mlflow

In [2]:
mlflow.set_tracking_uri("http://localhost:5000")
getml.engine.launch(allow_remote_ips=True, launch_browser=False, token="TOKEN")
getml_mlflow.autolog()

Launching ./getML --allow-push-notifications=true --allow-remote-ips=true --home-directory=/home/manuel/.getML --in-memory=true --install=false --launch-browser=false --log=false --project-directory=/home/manuel/.getML/projects --token=TOKEN in /home/manuel/.getML/getml-enterprise-1.5.0-amd64-linux...
Launched the getML Engine. The log output will be stored in /home/manuel/.getML/logs/getml_20250206115557.log


In [3]:
getml.set_project("interstate94")

Output()

Connected to project 'interstate94'.

In [4]:
traffic = getml.datasets.load_interstate94(roles=False, units=False)

In [5]:
traffic.set_role("ds", getml.data.roles.time_stamp)
traffic.set_role("holiday", getml.data.roles.categorical)
traffic.set_role("traffic_volume", getml.data.roles.target)

In [6]:
split = getml.data.split.time(traffic, "ds", test=getml.data.time.datetime(2018, 3, 15))

In [7]:
time_series = getml.data.TimeSeries(
    population=traffic,
    split=split,
    time_stamps="ds",
    horizon=getml.data.time.hours(1),
    memory=getml.data.time.days(7),
    lagged_targets=True,
)

pipe = getml.pipeline.Pipeline(
    tags=["memory: 7d", "horizon: 1h", "fast_prop"],
    data_model=time_series.data_model,
    preprocessors=[getml.preprocessors.Seasonal()],
    feature_learners=[
        getml.feature_learning.FastProp(
            loss_function=getml.feature_learning.loss_functions.SquareLoss,
            num_threads=1,
            num_features=20,
        )
    ],
    predictors=[getml.predictors.XGBoostRegressor()],
)
pipe

Pipeline(data_model='population',
         feature_learners=['FastProp'],
         feature_selectors=[],
         include_categorical=False,
         loss_function='SquareLoss',
         peripheral=['traffic'],
         predictors=['XGBoostRegressor'],
         preprocessors=['Seasonal'],
         share_selected_features=0.5,
         tags=['memory: 7d', 'horizon: 1h', 'fast_prop'])

In [8]:
traffic.roles.to_dict()

{'categorical': ['holiday'],
 'join_key': [],
 'numerical': [],
 'target': ['traffic_volume'],
 'text': [],
 'time_stamp': ['ds'],
 'unused_float': ['hour', 'weekday', 'day', 'month', 'year'],
 'unused_string': []}

In [9]:
fit1 = pipe.fit(time_series.train)
print(fit1.id)

Checking data model...

Output()

OK.

Output()

Trained pipeline.

2025/02/06 11:56:07 INFO mlflow.tracking._tracking_service.client: 🏃 View run fit at: http://localhost:5000/#/experiments/844494434965818253/runs/1884e43fd9044933ac2167759294bd39.
2025/02/06 11:56:07 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: http://localhost:5000/#/experiments/844494434965818253.
2025/02/06 11:56:07 INFO mlflow.tracking._tracking_service.client: 🏃 View run Pipeline-bwXFsw at: http://localhost:5000/#/experiments/844494434965818253/runs/9e715947f2494764adc405c2e6514e67.
2025/02/06 11:56:07 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: http://localhost:5000/#/experiments/844494434965818253.


Time taken: 0:00:08.803014.

bwXFsw


In [10]:
fit2 = pipe.fit(time_series.train)
print(fit2.id)

Checking data model...

Output()

OK.

Output()

Trained pipeline.

2025/02/06 11:56:08 INFO mlflow.tracking._tracking_service.client: 🏃 View run fit at: http://localhost:5000/#/experiments/844494434965818253/runs/9158e92f77d742a9aa324efd23de8778.
2025/02/06 11:56:08 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: http://localhost:5000/#/experiments/844494434965818253.
2025/02/06 11:56:08 INFO mlflow.tracking._tracking_service.client: 🏃 View run Pipeline-VsF9mj at: http://localhost:5000/#/experiments/844494434965818253/runs/f9bff4f208f2429cae7d7d6619030a0a.
2025/02/06 11:56:08 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: http://localhost:5000/#/experiments/844494434965818253.


Time taken: 0:00:00.292132.

VsF9mj


In [11]:
pipe.score(time_series.test)

Output()

2025/02/06 11:56:09 INFO mlflow.tracking._tracking_service.client: 🏃 View run score at: http://localhost:5000/#/experiments/844494434965818253/runs/5f9a9436c3114dcab4610355f7962fb1.
2025/02/06 11:56:09 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: http://localhost:5000/#/experiments/844494434965818253.


,date time,set used,target,mae,rmse,rsquared
0,2025-02-06 11:56:08,train,traffic_volume,200.4302,299.2045,0.9768
1,2025-02-06 11:56:09,test,traffic_volume,179.9515,269.631,0.9816


In [12]:
pipe.predict(population_table=traffic, peripheral_tables=[traffic])

Output()

2025/02/06 11:56:09 INFO mlflow.tracking._tracking_service.client: 🏃 View run predict at: http://localhost:5000/#/experiments/844494434965818253/runs/c9daf404156e4e72ab6641034e331f5d.
2025/02/06 11:56:09 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: http://localhost:5000/#/experiments/844494434965818253.


array([[ 592.84667969],
       [1517.38391113],
       [1517.38391113],
       ...,
       [2516.60766602],
       [1748.44812012],
       [1240.2791748 ]])

In [13]:
pipe.transform(population_table=traffic, peripheral_tables=[traffic])

Output()

2025/02/06 11:56:10 INFO mlflow.tracking._tracking_service.client: 🏃 View run transform at: http://localhost:5000/#/experiments/844494434965818253/runs/5749f68c1d4d44e7b7cefe817ac2c7ac.
2025/02/06 11:56:10 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: http://localhost:5000/#/experiments/844494434965818253.


array([[     0.        ,      0.        ,      0.        , ...,
             0.        ,      0.        ,      0.        ],
       [  1513.        ,   1513.        ,      0.        , ...,
          3600.        ,      0.        ,      0.        ],
       [  1513.        ,   1550.        ,      0.        , ...,
          5400.        ,   1800.        ,   1800.        ],
       ...,
       [  2346.        ,   2781.        ,    458.71528674, ...,
        109800.        , 186850.63553545, 106200.        ],
       [  1635.        ,   2159.        ,    472.51316794, ...,
         88200.        , 156751.65070901,  84600.        ],
       [   934.        ,   1450.        ,   1216.33670486, ...,
         66600.        , 114631.40930827,  63000.        ]])

In [14]:
pipe.transform(population_table=traffic, peripheral_tables=[traffic], df_name="transformed_df")

Output()

2025/02/06 11:56:10 INFO mlflow.tracking._tracking_service.client: 🏃 View run transform at: http://localhost:5000/#/experiments/844494434965818253/runs/7e0d71c8470942a99f17edbbc95e162b.
2025/02/06 11:56:10 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: http://localhost:5000/#/experiments/844494434965818253.


name,ds,traffic_volume,feature_1_1,feature_1_2,feature_1_3,feature_1_4,feature_1_5,feature_1_6,feature_1_7,feature_1_8,feature_1_9,feature_1_10,feature_1_11,feature_1_12,feature_1_13,feature_1_14,feature_1_15,feature_1_16,feature_1_17,feature_1_18,feature_1_19,feature_1_20
role,time_stamp,target,numerical,numerical,numerical,numerical,numerical,numerical,numerical,numerical,numerical,numerical,numerical,numerical,numerical,numerical,numerical,numerical,numerical,numerical,numerical,numerical
unit,"time stamp, comparison only",,,,,,,,,,,,,,,,,,,,,
0,2016-01-01,1513,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
1,2016-01-01 01:00:00,1550,1513,1513,0,0,0,0,0,0,0,0,0,0,1513,1513,1513,1513,0,3600,0,0
2,2016-01-01 02:00:00,993,1513,1550,0,0,0,0,0,0,0,0,0,0,1513,1550,1513,1550,1800,5400,1800,1800
3,2016-01-01 03:00:00,719,1513,993,0,0,0,0,0,0,0,0,0,0,1513,993,1513,993,2939.3877,7200,2939.3877,3600
4,2016-01-01 04:00:00,533,1513,719,0,0,0,0,0,0,0,0,0,0,1513,719,1513,719,4024.9224,9000,4024.9224,5400
,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
24091,2018-09-30 19:00:00,3543,3516,3947,371.7278,24989,3569.8571,3058,3516,3818,4286,3058,3516,4041.0402,3516,3947,3516,3947,228760.5735,153000,228760.5735,149400
24092,2018-09-30 20:00:00,2781,2846,3543,324.0932,21099,3014.1429,2559,2846,3338,3538,2559,2856,3512.4688,2846,3543,2846,3543,210158.5116,131400,210158.5116,127800
